In [2]:
# rescuing variants with high depth and the ones that were flagged only because of strandoddsratio

import pandas as pd

def rescue_variants(input_file, output_file):
    # Load the variants file
    df = pd.read_csv(input_file)
    
    # Define your "High Quality" thresholds for rescue
    DEPTH_THRESHOLD = 100 # Using a higher bar for SOR rescue
    AF_LOWER = 0.40       # Heterozygous range (~0.5)
    AF_UPPER = 0.60
    HOM_AF_THRESHOLD = 0.90 # Homozygous range (~1.0)

    def check_row(row):
        # We only care about rows currently marked 'low'
        if str(row['Confidence']).lower() != 'low':
            return row['Confidence']
        
        # 1. Identify the Filter Status
        # We accept 'PASS' OR 'StrandOddsRatio'
        current_filter = str(row['Filters 1']).upper()
        is_acceptable_filter = (current_filter == 'PASS' or current_filter == 'STRANDODDSRATIO')
        
        # 2. Check Depth 1 (The primary tube)
        # We want to be very sure, so we check if it meets a solid depth
        has_depth = row['Depth 1'] >= DEPTH_THRESHOLD
        
        # 3. Check Allele Fraction (Biological consistency)
        af = row['Allele fraction 1']
        is_het = (af >= AF_LOWER and af <= AF_UPPER)
        is_hom = (af >= HOM_AF_THRESHOLD)
        
        # 4. Check Replicate Status
        # Check if replicates 2-6 are empty/zero (the "lonely tube" scenario)
        other_reps_depth = sum([row.get(f'Depth {i}', 0) for i in range(2, 7) if pd.notnull(row.get(f'Depth {i}'))])

        # RESCUE LOGIC:
        # If (Filter is PASS or SOR) AND (High Depth) AND (50% or 100% AF)
        if is_acceptable_filter and has_depth and (is_het or is_hom):
            
            # Sub-label for clarity
            if current_filter == 'STRANDODDSRATIO':
                return 'high (rescued-SOR)'
            elif other_reps_depth == 0:
                return 'high (rescued-SingleRep)'
            else:
                return 'high (rescued)'
        
        return row['Confidence']

    # Apply the logic
    print(f"Processing {input_file}...")
    df['Confidence'] = df.apply(check_row, axis=1)
    
    # Summary of changes
    total_rescued = len(df[df['Confidence'].str.contains('rescued', na=False)])
    sor_rescued = len(df[df['Confidence'] == 'high (rescued-SOR)'])
    
    df.to_csv(output_file, index=False)
    print(f"Done! Rescued {total_rescued} variants total.")
    print(f"Specifically rescued {sor_rescued} variants that failed ONLY on StrandOddsRatio.")
    print(f"Results saved to: {output_file}")

# Usage
if __name__ == "__main__":
    rescue_variants('variants.csv', 'variants_final_rescued.csv')

Processing variants.csv...
Done! Rescued 297 variants total.
Specifically rescued 74 variants that failed ONLY on StrandOddsRatio.
Results saved to: variants_final_rescued.csv


In [3]:
# how many variants i have now
import pandas as pd

# Load your file
df = pd.read_csv('variants_final_rescued.csv')

# Count rows where Confidence is NOT 'low' (case-insensitive)
non_low_count = df[df['Confidence'].str.lower() != 'low'].shape[0]

print(f"Total non-low variants: {non_low_count}")

Total non-low variants: 367


In [6]:
# how many varints for each sample if i didn't rescue and if i do
import pandas as pd

def process_variants(input_file):
    # Load the variants file
    # If your file is Excel, use pd.read_excel(input_file)
    df = pd.read_csv(input_file)
    
    # 1. Filter out all 'low' confidence variants
    # Using str.lower() to ensure it catches 'Low', 'LOW', or 'low'
    filtered_df = df[df['Confidence'].str.lower() != 'low'].copy()
    
    # 2. Print the total count of variants kept
    print(f"Total non-low variants kept: {len(filtered_df)}")
    print("-" * 30)
    
    # 3. Print the number of rows for each unique sample
    # This uses the 'Sample' column to group and count
    sample_counts = filtered_df['Sample'].value_counts()
    
    print("Variant counts per sample (High/Medium only):")
    print(sample_counts.to_string())
    
    # 4. Optional: Save the cleaned table
    # filtered_df.to_csv('variants_filtered.csv', index=False)

if __name__ == "__main__":
    # Change 'variants.csv' to your actual file name
    process_variants('variants.csv')

if __name__ == "__main__":
    # Change 'variants.csv' to your actual file name
    process_variants('variants_final_rescued.csv')

Total non-low variants kept: 70
------------------------------
Variant counts per sample (High/Medium only):
Sample
control14            4
case1                3
control1             3
case8                3
case7                3
case3                3
case5                3
control3             3
control8             3
control5             3
control4             2
control2             2
PAH-29_S19_          2
case6                2
case10               2
PAH-21_S11_          2
control9             2
control7             2
case4                2
PAH-17_S7_           1
PAH-11_S1_           1
PAH-14_S4_           1
Contr-PAH-15_S30_    1
PAH-15_S5_           1
PAH-20_S10_          1
PAH-19_S9_           1
PAH-31_S21_          1
PAH-32_S22_          1
PAH-33_S23_          1
PAH-34_S24_          1
PAH-36_S26_          1
PAH-37_S27_          1
case2                1
PAH-26_S16_          1
control13            1
control10            1
case9                1
control12            1
control11 

In [19]:
# whats the difference between controls and samples if we use different filters
# i have 52 samples with 38 cases and 14 controls. none of the filters include all cases and no controls but at stingent filtering there are only cases (though not all) with only pathogenic variants 
import pandas as pd

def process_variants(input_file):
    # Load the variants file
    # If your file is Excel, use pd.read_excel(input_file)
    df = pd.read_csv(input_file)
    
    # 1. Filter out all 'low' confidence variants
    # Using str.lower() to ensure it catches 'Low', 'LOW', or 'low'
    filtered_df = df[
        #(df['Confidence'].str.lower() != 'low') & 
        #(df['Impact'].str.lower() != 'low') & 
        #(df['Consequence'].str.lower() != 'intron') &
        (df['Clinical significance'].str.lower() == 'pathogenic')
    ].copy() 
    
    # 2. Print the total count of variants kept
    print(f"Total non-low variants kept: {len(filtered_df)}")
    print("-" * 30)
    
    # 3. Print the number of rows for each unique sample
    # This uses the 'Sample' column to group and count
    sample_counts = filtered_df['Sample'].value_counts()
    
    print("Variant counts per sample (High/Medium only):")
    print(sample_counts.to_string())
    
    # 4. Optional: Save the cleaned table
    # filtered_df.to_csv('variants_filtered.csv', index=False)

if __name__ == "__main__":
    # Change 'variants.csv' to your actual file name
    process_variants('variants.csv')

if __name__ == "__main__":
    # Change 'variants.csv' to your actual file name
#    process_variants('variants_final_rescued.csv')

Total non-low variants kept: 36
------------------------------
Variant counts per sample (High/Medium only):
Sample
PAH-35_S25_    3
PAH-31_S21_    3
PAH-22_S12_    2
PAH-13_S3_     2
PAH-12_S2_     2
PAH-36_S26_    2
PAH-37_S27_    2
PAH-33_S23_    2
PAH-25_S15_    2
PAH-30_S20_    2
PAH-32_S22_    2
PAH-26_S16_    2
PAH-27_S17_    2
PAH-18_S8_     1
PAH-16_S6_     1
PAH-14_S4_     1
PAH-19_S9_     1
PAH-29_S19_    1
PAH-28_S18_    1
case5          1
case6          1
Total non-low variants kept: 36
------------------------------
Variant counts per sample (High/Medium only):
Sample
PAH-35_S25_    3
PAH-31_S21_    3
PAH-22_S12_    2
PAH-13_S3_     2
PAH-12_S2_     2
PAH-36_S26_    2
PAH-37_S27_    2
PAH-33_S23_    2
PAH-25_S15_    2
PAH-30_S20_    2
PAH-32_S22_    2
PAH-26_S16_    2
PAH-27_S17_    2
PAH-18_S8_     1
PAH-16_S6_     1
PAH-14_S4_     1
PAH-19_S9_     1
PAH-29_S19_    1
PAH-28_S18_    1
case5          1
case6          1


In [38]:
# how many variants are called in controls under different filters

def filter_out_control_variants(output_file):
    
    # 1. Define how we uniquely identify a variant
    # Typically: Chromosome + Position + Ref + Alt
    variant_cols = ['Chromosome', 'Position', 'Ref', 'Alt']
    
    # 2. Identify Control Samples
    # We look for rows where the 'Sample' name contains 'Contr' (case-insensitive)
    is_control = df['Sample'].str.contains('Contr', case=False, na=False)
    control_df = df[is_control]
    
    # 3. Create a list of unique variants found in Controls
    # We drop duplicates to get a unique set of coordinates/alleles
    control_variants = control_df[variant_cols].drop_duplicates()
    
    # 4. Filter the main dataframe
    # We want to keep rows that are NOT in the control_variants set.
    # We use a 'left join' with an indicator to find non-matches.
    merged = df.merge(control_variants, on=variant_cols, how='left', indicator=True)
    
    # Rows where _merge is 'left_only' are variants NOT found in any control sample
    filtered_df = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])
    
    # 5. Save and Print Results
    filtered_df.to_csv(output_file, index=False)
    
    print(f"Original total rows: {len(df)}")
    print(f"Original unique variants: {df[variant_cols].drop_duplicates().shape[0]}")
    print(f"Unique variants found in Controls: {len(control_variants)}")
    print(f"Rows remaining after removing Control-linked variants: {len(filtered_df)}")
    print(f"Remaining unique variants: {filtered_df[variant_cols].drop_duplicates().shape[0]}")


# before any filtering
if __name__ == "__main__":
    print("\n before any filtering")
    import pandas as pd
    df = pd.read_csv('variants.csv')
    
    # select rows where Confidence is NOT 'low' (case-insensitive)
    #df = df[df['Confidence'].str.lower() != 'low']
    filter_out_control_variants('variants_no_controls.csv')

# after original filters
if __name__ == "__main__":
    print("\n after original filters")

    import pandas as pd
    df = pd.read_csv('variants.csv')
    
    # select rows where Confidence is NOT 'low' (case-insensitive)
    df = df[df['Confidence'].str.lower() != 'low']
    filter_out_control_variants('variants_no_controls.csv')

# after rescuing
if __name__ == "__main__":
    print("\n after rescuing")
    import pandas as pd
    df = pd.read_csv('variants_final_rescued.csv')
    
    # select rows where Confidence is NOT 'low' (case-insensitive)
    df = df[df['Confidence'].str.lower() != 'low']
    filter_out_control_variants('variants_no_controls.csv')


 before any filtering
Original total rows: 545
Original unique variants: 45
Unique variants found in Controls: 23
Rows remaining after removing Control-linked variants: 71
Remaining unique variants: 22

 after original filters
Original total rows: 70
Original unique variants: 12
Unique variants found in Controls: 8
Rows remaining after removing Control-linked variants: 16
Remaining unique variants: 4

 after rescuing
Original total rows: 367
Original unique variants: 31
Unique variants found in Controls: 16
Rows remaining after removing Control-linked variants: 54
Remaining unique variants: 15


In [19]:
# saving all good variants after rescuing (includes multiple identifiers) for vep
import pandas as pd
df = pd.read_csv('variants_final_rescued.csv')
variant_cols = ['Chromosome', 'Position', 'Ref', 'Alt','Existing variation']
unique_variants = df[variant_cols].drop_duplicates()
df1 = unique_variants[['Existing variation']].copy()
df1['Existing variation'] = df1['Existing variation'].str.split(', ') # Step 1: Split into lists
df1 = df1.explode('Existing variation')                 # Step 2: Explode the lists
df1.dropna().to_csv("rsids.csv",index=False,header=False)
df1.dropna()

,Existing variation
0,rs772897
0,COSV108133564
2,rs1522306
2,COSV61015928
3,rs1042503
...,...
248,CM930536
260,rs118092776
260,CM981427
260,COSV61020094
